# 🩺 Health Monitoring System (Diploma, PyTorch)

**Тема:** распознавание 12 видов активности человека по данным wearable-сенсоров (MHEALTH).

Ноутбук оформлен как единый production-like pipeline:
1. загрузка и проверка датасета;
2. подготовка оконных выборок без утечки по субъектам;
3. обучение CNN-LSTM на PyTorch;
4. оценка качества и визуализация;
5. модуль аномалий и экспорт артефактов.

In [ ]:
# Unified environment setup (Colab/local)
!pip -q install numpy pandas scikit-learn seaborn matplotlib joblib torch

In [ ]:
# Notes
# - Ноутбук структурирован как production-like pipeline с проверками и артефактами.
# - Рекомендуемый запуск: сверху вниз без пропуска ячеек.
# - Все ключевые результаты сохраняются в ./artifacts.

In [ ]:
# Auto environment check (CUDA/CPU, versions, dataset files)
import sys
import importlib
from pathlib import Path

packages = ['numpy', 'pandas', 'sklearn', 'seaborn', 'matplotlib', 'joblib', 'torch']
print('Python:', sys.version.split()[0])

for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{pkg}: {ver}')
    except Exception as e:
        print(f'{pkg}: not available ({e})')

try:
    import torch
    print('Torch CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch CUDA check error:', e)

mhealth_dir = Path('./mhealth_data')
zip_path = mhealth_dir / 'mhealth.zip'
subject_files = list(mhealth_dir.rglob('mHealth_subject*.log')) if mhealth_dir.exists() else []

print('Dataset dir exists:', mhealth_dir.exists())
print('Zip exists:', zip_path.exists())
print('Subject files found:', len(subject_files))
if len(subject_files) > 0:
    print('First subject file:', subject_files[0])

In [ ]:
# Project configuration and strict sanity checks
from dataclasses import dataclass

@dataclass
class Config:
    window: int = 128
    step: int = 64
    batch_size: int = 128
    num_epochs: int = 80
    learning_rate: float = 1e-3
    patience: int = 10
    grad_clip_norm: float = 1.0

CFG = Config()
print(CFG)

assert CFG.window > 0 and CFG.step > 0, 'Window and step must be positive.'
assert CFG.step <= CFG.window, 'STEP should not exceed WINDOW for overlapped segmentation.'

In [ ]:
# Main training pipeline starts below

## Diploma Version: Advanced Activity Recognition (12 classes, PyTorch)

Этот блок переведен на PyTorch и оформлен в едином стиле дипломного проекта:
- надежная загрузка и проверка MHEALTH;
- явное отображение 12 классов (EN + RU);
- оконная сегментация и split по субъектам без утечек;
- обучение CNN-LSTM с class weights, early stopping и scheduler;
- подробная оценка качества (Accuracy, Weighted F1, отчет по классам, confusion matrix);
- модуль аномалий и экспорт артефактов для последующего использования.

In [ ]:
# Dependencies were already installed above (cell 1).

In [ ]:
import os
import glob
import json
import zipfile
import urllib.request
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_trainable_parameters(model_obj) -> int:
    return sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

# Data sources and paths
DATA_DIR = Path('./mhealth_data')
ZIP_URL = 'https://archive.ics.uci.edu/static/public/319/mhealth+dataset.zip'
ZIP_PATH = DATA_DIR / 'mhealth.zip'

# Official MHEALTH activity labels (excluding transition class 0)
ACTIVITY_NAMES = {
    1: 'Standing still',
    2: 'Sitting and relaxing',
    3: 'Lying down',
    4: 'Walking',
    5: 'Climbing stairs',
    6: 'Waist bends forward',
    7: 'Frontal elevation of arms',
    8: 'Knees bending (crouching)',
    9: 'Cycling',
    10: 'Jogging',
    11: 'Running',
    12: 'Jumping front and back'
}

ACTIVITY_NAMES_RU = {
    1: 'Стоя неподвижно',
    2: 'Сидя в расслабленном состоянии',
    3: 'Лежа',
    4: 'Ходьба',
    5: 'Подъем по лестнице',
    6: 'Наклоны корпуса вперед',
    7: 'Подъем рук перед собой',
    8: 'Сгибание коленей (приседание)',
    9: 'Езда на велосипеде',
    10: 'Бег трусцой',
    11: 'Бег',
    12: 'Прыжки вперед-назад'
}


def ensure_mhealth_dataset(data_dir: Path, zip_path: Path, url: str):
    data_dir.mkdir(parents=True, exist_ok=True)

    if not zip_path.exists():
        print('Downloading MHEALTH dataset...')
        urllib.request.urlretrieve(url, zip_path)

    subject_files_local = sorted(glob.glob(str(data_dir / '**' / 'mHealth_subject*.log'), recursive=True))
    if subject_files_local:
        return subject_files_local

    print('Extracting MHEALTH archive...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)

    subject_files_local = sorted(glob.glob(str(data_dir / '**' / 'mHealth_subject*.log'), recursive=True))
    if not subject_files_local:
        raise FileNotFoundError('MHEALTH subject files were not found after extraction.')

    return subject_files_local


subject_files = ensure_mhealth_dataset(DATA_DIR, ZIP_PATH, ZIP_URL)
print(f'Found {len(subject_files)} subject files')

In [ ]:
def load_mhealth_subject_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t', header=None).dropna()
    n_cols = df.shape[1]
    feature_cols_local = [f'f_{i:02d}' for i in range(n_cols - 1)]
    df.columns = feature_cols_local + ['activity']

    file_name = os.path.basename(path)
    subject_id = int(file_name.split('subject')[1].split('.log')[0])
    df['subject'] = subject_id
    return df


frames = [load_mhealth_subject_file(p) for p in subject_files]
raw_df = pd.concat(frames, axis=0, ignore_index=True)

# Keep only 12 target activities (1..12), remove transition class 0
raw_df = raw_df[raw_df['activity'].between(1, 12)].copy()
raw_df['activity'] = raw_df['activity'].astype(int)

feature_cols = [c for c in raw_df.columns if c not in ['activity', 'subject']]

raw_df['activity_name_en'] = raw_df['activity'].map(ACTIVITY_NAMES)
raw_df['activity_name_ru'] = raw_df['activity'].map(ACTIVITY_NAMES_RU)

print('Raw shape:', raw_df.shape)
print('Subjects:', sorted(raw_df['subject'].unique()))
print('Activities count:', raw_df['activity'].nunique())
print('Activity labels in data:', sorted(raw_df['activity'].unique()))

raw_df[['activity', 'activity_name_en', 'activity_name_ru', 'subject']].head()

In [ ]:
# EDA: class balance and subject coverage
class_counts = raw_df['activity'].value_counts().sort_index()
class_names_ru = [ACTIVITY_NAMES_RU[k] for k in class_counts.index]

plt.figure(figsize=(12, 4))
sns.barplot(x=class_names_ru, y=class_counts.values, palette='viridis')
plt.title('Распределение выборки по видам активности')
plt.xlabel('Активность')
plt.ylabel('Количество сэмплов')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_activity_table = pd.crosstab(raw_df['subject'], raw_df['activity'])
print('Subject x Activity table shape:', subject_activity_table.shape)
plt.figure(figsize=(12, 6))
sns.heatmap(subject_activity_table, cmap='YlGnBu')
plt.title('Покрытие активностей по субъектам')
plt.xlabel('Класс активности')
plt.ylabel('Subject ID')
plt.tight_layout()
plt.show()

In [ ]:
# Data quality checks before windowing
assert raw_df[feature_cols].isnull().sum().sum() == 0, 'NaN detected in feature columns.'
assert raw_df['activity'].between(1, 12).all(), 'Unexpected activity labels found.'

summary_stats = raw_df[feature_cols].describe().T[['mean', 'std', 'min', 'max']].head(8)
print('Feature summary sample:')
print(summary_stats)

print('All quality checks passed.')

In [ ]:
WINDOW = CFG.window
STEP = CFG.step
BATCH_SIZE = CFG.batch_size
NUM_EPOCHS = CFG.num_epochs
LEARNING_RATE = CFG.learning_rate
PATIENCE = CFG.patience


def make_windows(df: pd.DataFrame, window: int = 128, step: int = 64):
    X_list, y_list, g_list = [], [], []

    for subject_id, sdf in df.groupby('subject'):
        sdf = sdf.reset_index(drop=True)
        X = sdf[feature_cols].values
        y = sdf['activity'].values

        for i in range(0, len(sdf) - window + 1, step):
            xw = X[i:i + window]
            yw = y[i:i + window]
            label = np.bincount(yw).argmax()
            X_list.append(xw)
            y_list.append(label)
            g_list.append(subject_id)

    return np.asarray(X_list), np.asarray(y_list), np.asarray(g_list)


X_win, y_win, groups = make_windows(raw_df, WINDOW, STEP)

classes = sorted(np.unique(y_win))
if len(classes) != 12:
    print(f'Warning: expected 12 classes, but got {len(classes)} classes: {classes}')

label_to_idx = {lab: i for i, lab in enumerate(classes)}
idx_to_label = {i: lab for lab, i in label_to_idx.items()}
idx_to_name_en = {i: ACTIVITY_NAMES[lab] for i, lab in idx_to_label.items()}
idx_to_name_ru = {i: ACTIVITY_NAMES_RU[lab] for i, lab in idx_to_label.items()}

y_idx = np.array([label_to_idx[v] for v in y_win], dtype=np.int64)

print('Windowed X shape:', X_win.shape)
print('Windowed y shape:', y_idx.shape)
print('Number of classes:', len(classes))

# Split by subject: Train+Val vs Test
gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
trainval_idx, test_idx = next(gss_1.split(X_win, y_idx, groups=groups))

X_trainval, X_test = X_win[trainval_idx], X_win[test_idx]
y_trainval, y_test = y_idx[trainval_idx], y_idx[test_idx]
g_trainval = groups[trainval_idx]

# Split Train+Val into Train and Val
gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx_local, val_idx_local = next(gss_2.split(X_trainval, y_trainval, groups=g_trainval))

X_train, X_val = X_trainval[train_idx_local], X_trainval[val_idx_local]
y_train, y_val = y_trainval[train_idx_local], y_trainval[val_idx_local]

# Fit scaler only on train set
n_features = X_train.shape[-1]
scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)
X_val_2d = X_val.reshape(-1, n_features)
X_test_2d = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val_2d).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)

# Convert to PyTorch format: [N, C, T]
X_train_t = torch.tensor(np.transpose(X_train_scaled, (0, 2, 1)), dtype=torch.float32)
X_val_t = torch.tensor(np.transpose(X_val_scaled, (0, 2, 1)), dtype=torch.float32)
X_test_t = torch.tensor(np.transpose(X_test_scaled, (0, 2, 1)), dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_t = torch.tensor(class_weights_arr, dtype=torch.float32, device=DEVICE)

print(f'Train windows: {X_train_scaled.shape[0]} | Val windows: {X_val_scaled.shape[0]} | Test windows: {X_test_scaled.shape[0]}')
print('Train subjects:', sorted(np.unique(g_trainval[train_idx_local])))
print('Val subjects:', sorted(np.unique(g_trainval[val_idx_local])))
print('Test subjects:', sorted(np.unique(groups[test_idx])))

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, n_features: int, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.2)
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return self.classifier(x)


def run_epoch(model_local, loader, criterion, optimizer=None, grad_clip_norm=None):
    is_train = optimizer is not None
    model_local.train() if is_train else model_local.eval()

    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            logits = model_local(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model_local.parameters(), max_norm=grad_clip_norm)
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(yb.detach().cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, acc, y_true, y_pred


model = CNNLSTM(n_features=n_features, n_classes=len(classes)).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

os.makedirs('./artifacts', exist_ok=True)
checkpoint_path = './artifacts/best_activity_model.pt'

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
best_val_loss = float('inf')
best_epoch = -1
patience_counter = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, grad_clip_norm=CFG.grad_clip_norm
    )
    val_loss, val_acc, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(float(optimizer.param_groups[0]['lr']))

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        patience_counter += 1

    current_lr = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:03d}/{NUM_EPOCHS} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | '
        f'lr={current_lr:.6f}'
    )

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

# Load best checkpoint before test
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
training_time_sec = time.time() - start_time

test_loss, test_acc, y_test_true, y_test_pred = run_epoch(model, test_loader, criterion, optimizer=None)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Best epoch: {best_epoch} | Best val loss: {best_val_loss:.4f}')
print(f'Training time: {training_time_sec:.1f} sec')
print(f'Best checkpoint saved to: {checkpoint_path}')
print(f'Trainable params: {count_trainable_parameters(model):,}')

In [ ]:
acc = accuracy_score(y_test_true, y_test_pred)
f1_w = f1_score(y_test_true, y_test_pred, average='weighted')

print(f'Test Accuracy: {acc:.4f}')
print(f'Weighted F1:  {f1_w:.4f}')

target_names_ru = [idx_to_name_ru[i] for i in range(len(classes))]
print(classification_report(y_test_true, y_test_pred, target_names=target_names_ru, digits=4))

cm = confusion_matrix(y_test_true, y_test_pred)
plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    cmap='Blues',
    annot=False,
    xticklabels=target_names_ru,
    yticklabels=target_names_ru
)
plt.title('Матрица ошибок (12 видов активности)')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

history_df = pd.DataFrame(history)
history_df.to_csv('./artifacts/training_history.csv', index=False)

plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='val loss')
plt.legend()
plt.title('Динамика функции потерь')

plt.subplot(1, 3, 2)
plt.plot(history['train_acc'], label='train acc')
plt.plot(history['val_acc'], label='val acc')
plt.legend()
plt.title('Динамика точности')

plt.subplot(1, 3, 3)
plt.plot(history['lr'], label='learning rate', color='purple')
plt.legend()
plt.title('Динамика learning rate')

plt.tight_layout()
plt.show()

print('Saved training history to ./artifacts/training_history.csv')

In [ ]:
# Extended evaluation: per-class metrics and top confusing pairs
report_dict = classification_report(y_test_true, y_test_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).T
report_df.to_csv('./artifacts/classification_report_detailed.csv', index=True)

macro_precision = precision_score(y_test_true, y_test_pred, average='macro')
macro_recall = recall_score(y_test_true, y_test_pred, average='macro')
print(f'Macro Precision: {macro_precision:.4f}')
print(f'Macro Recall:    {macro_recall:.4f}')

cm = confusion_matrix(y_test_true, y_test_pred)
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)

pairs = []
for i in range(cm_offdiag.shape[0]):
    for j in range(cm_offdiag.shape[1]):
        if cm_offdiag[i, j] > 0:
            pairs.append((i, j, int(cm_offdiag[i, j])))

pairs = sorted(pairs, key=lambda x: x[2], reverse=True)[:10]
print('Top-10 confusing class pairs (true -> pred):')
for i, j, c in pairs:
    print(f'- {idx_to_name_ru[i]} -> {idx_to_name_ru[j]}: {c}')

In [ ]:
# Real-time style simulation with anomaly flag
# This cell can be executed after model training. If anomaly model is unavailable, status defaults to N/A.
rng = np.random.default_rng(SEED)

print('Simulating real-time predictions...')
for step_id in range(1, 6):
    sample_idx = int(rng.integers(0, len(X_test)))
    sample_window = X_test[sample_idx]

    sample_2d = sample_window.reshape(-1, n_features)
    sample_scaled = scaler.transform(sample_2d).reshape(1, WINDOW, n_features)
    sample_t = torch.tensor(np.transpose(sample_scaled, (0, 2, 1)), dtype=torch.float32, device=DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(sample_t)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]

    cls_idx = int(np.argmax(probs))
    conf = float(np.max(probs))
    orig_lbl = idx_to_label[cls_idx]
    cls_name_ru = idx_to_name_ru[cls_idx]

    if 'iso' in globals():
        sample_stats = np.concatenate([
            X_test_scaled[sample_idx].mean(axis=0),
            X_test_scaled[sample_idx].std(axis=0),
            X_test_scaled[sample_idx].max(axis=0),
            X_test_scaled[sample_idx].min(axis=0)
        ], axis=0).reshape(1, -1)
        anomaly = int(iso.predict(sample_stats)[0] == -1)
        status = 'ALERT' if anomaly else 'OK'
    else:
        status = 'N/A'

    print(f'[{step_id}] class={orig_lbl} ({cls_name_ru}) | conf={conf:.3f} | anomaly={status}')

In [ ]:
# Experiment manifest for reproducibility
manifest = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'device': str(DEVICE),
    'seed': int(SEED),
    'window': int(WINDOW),
    'step': int(STEP),
    'batch_size': int(BATCH_SIZE),
    'epochs_configured': int(NUM_EPOCHS),
    'learning_rate': float(LEARNING_RATE),
    'n_features': int(n_features),
    'n_classes': int(len(classes)),
    'test_accuracy': float(acc),
    'test_f1_weighted': float(f1_w)
}

with open('./artifacts/experiment_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Saved manifest to ./artifacts/experiment_manifest.json')

In [ ]:
# Optional anomaly module (unsupervised) on window-level summary features
from sklearn.ensemble import IsolationForest

# Build compact statistical features from windows
X_test_stats = np.concatenate([
    X_test_scaled.mean(axis=1),
    X_test_scaled.std(axis=1),
    X_test_scaled.max(axis=1),
    X_test_scaled.min(axis=1)
], axis=1)

X_train_stats = np.concatenate([
    X_train_scaled.mean(axis=1),
    X_train_scaled.std(axis=1),
    X_train_scaled.max(axis=1),
    X_train_scaled.min(axis=1)
], axis=1)

iso = IsolationForest(
    n_estimators=300,
    contamination=0.03,
    random_state=SEED,
    n_jobs=-1
)
iso.fit(X_train_stats)

anomaly_flag = (iso.predict(X_test_stats) == -1).astype(int)
print('Detected anomaly windows:', anomaly_flag.sum(), 'of', len(anomaly_flag))

In [ ]:
# Save diploma-grade artifacts
import joblib

os.makedirs('./artifacts', exist_ok=True)
torch.save(model.state_dict(), './artifacts/activity_cnn_lstm_state_dict.pt')
joblib.dump(scaler, './artifacts/feature_scaler.pkl')
joblib.dump(
    {
        'classes': classes,
        'label_to_idx': label_to_idx,
        'idx_to_label': idx_to_label,
        'idx_to_name_en': idx_to_name_en,
        'idx_to_name_ru': idx_to_name_ru,
        'activity_names_en': ACTIVITY_NAMES,
        'activity_names_ru': ACTIVITY_NAMES_RU,
        'window': WINDOW,
        'step': STEP,
        'feature_cols': feature_cols,
        'seed': SEED,
        'n_features': n_features
    },
    './artifacts/label_mapping.pkl'
)
joblib.dump(iso, './artifacts/anomaly_isolation_forest.pkl')


# Inference helpers for deployment scripts

def _predict_proba_window(window_data: np.ndarray) -> np.ndarray:
    if window_data.shape != (WINDOW, n_features):
        raise ValueError(f'Expected shape {(WINDOW, n_features)}, got {window_data.shape}')

    sample_2d = window_data.reshape(-1, n_features)
    sample_scaled = scaler.transform(sample_2d).reshape(1, WINDOW, n_features)
    sample_t = torch.tensor(np.transpose(sample_scaled, (0, 2, 1)), dtype=torch.float32, device=DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(sample_t)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    return probs


def predict_activity(window_data: np.ndarray, confidence_threshold: float = 0.60):
    """
    Returns structured prediction with confidence gating.
    """
    probs = _predict_proba_window(window_data)
    class_idx = int(np.argmax(probs))
    conf = float(np.max(probs))

    original_label = idx_to_label[class_idx]
    readable_name_ru = idx_to_name_ru[class_idx]
    status = 'CONFIDENT' if conf >= confidence_threshold else 'LOW_CONFIDENCE'

    return {
        'class_idx': class_idx,
        'label': original_label,
        'name_ru': readable_name_ru,
        'confidence': conf,
        'status': status
    }


def predict_top_k(window_data: np.ndarray, k: int = 3):
    probs = _predict_proba_window(window_data)
    order = np.argsort(probs)[::-1][:k]
    return [
        {
            'rank': rank + 1,
            'class_idx': int(i),
            'label': int(idx_to_label[int(i)]),
            'name_ru': idx_to_name_ru[int(i)],
            'probability': float(probs[int(i)])
        }
        for rank, i in enumerate(order)
    ]


def predict_from_dataframe(df_chunk: pd.DataFrame, confidence_threshold: float = 0.60):
    arr = df_chunk[feature_cols].values
    if len(arr) < WINDOW:
        raise ValueError(f'Need at least {WINDOW} rows, got {len(arr)}')

    results = []
    for start in range(0, len(arr) - WINDOW + 1, STEP):
        w = arr[start:start + WINDOW]
        pred = predict_activity(w, confidence_threshold=confidence_threshold)
        pred['start_idx'] = int(start)
        pred['end_idx'] = int(start + WINDOW - 1)
        results.append(pred)
    return pd.DataFrame(results)


def predict_from_csv(csv_path: str, confidence_threshold: float = 0.60):
    df_in = pd.read_csv(csv_path)
    return predict_from_dataframe(df_in, confidence_threshold=confidence_threshold)


example_window = X_test[0]
pred_one = predict_activity(example_window, confidence_threshold=0.60)
pred_top3 = predict_top_k(example_window, k=3)
print('Single prediction:', pred_one)
print('Top-3:', pred_top3)
print('Saved artifacts to ./artifacts/')

## WOW Block: ROC/PR, Calibration, Baseline vs CNN-LSTM

Ниже добавлен расширенный сравнительный анализ:
- CNN-LSTM (основная модель);
- baseline-модель (One-vs-Rest Logistic Regression на статистических признаках окна);
- ROC/PR-кривые one-vs-rest;
- calibration-анализ уверенности вероятностей.

In [ ]:
# Build probabilities for CNN-LSTM and baseline, then compare metrics

def build_window_stats(X_scaled_windows: np.ndarray) -> np.ndarray:
    return np.concatenate([
        X_scaled_windows.mean(axis=1),
        X_scaled_windows.std(axis=1),
        X_scaled_windows.max(axis=1),
        X_scaled_windows.min(axis=1)
    ], axis=1)

# CNN-LSTM probabilities on test set
model.eval()
all_logits = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        all_logits.append(logits.detach().cpu().numpy())

logits_test = np.vstack(all_logits)
cnn_proba = np.exp(logits_test - logits_test.max(axis=1, keepdims=True))
cnn_proba = cnn_proba / cnn_proba.sum(axis=1, keepdims=True)
cnn_pred = np.argmax(cnn_proba, axis=1)

# Baseline: OVR logistic regression on compact statistical window features
X_train_stats = build_window_stats(X_train_scaled)
X_test_stats = build_window_stats(X_test_scaled)

baseline_clf = OneVsRestClassifier(
    LogisticRegression(max_iter=1500, class_weight='balanced', random_state=SEED)
)
baseline_clf.fit(X_train_stats, y_train)
baseline_proba = baseline_clf.predict_proba(X_test_stats)
baseline_pred = np.argmax(baseline_proba, axis=1)

# Common metrics
comparison_df = pd.DataFrame([
    {
        'model': 'CNN-LSTM (PyTorch)',
        'accuracy': accuracy_score(y_test, cnn_pred),
        'f1_weighted': f1_score(y_test, cnn_pred, average='weighted'),
        'precision_macro': precision_score(y_test, cnn_pred, average='macro'),
        'recall_macro': recall_score(y_test, cnn_pred, average='macro')
    },
    {
        'model': 'Baseline OVR LogisticRegression',
        'accuracy': accuracy_score(y_test, baseline_pred),
        'f1_weighted': f1_score(y_test, baseline_pred, average='weighted'),
        'precision_macro': precision_score(y_test, baseline_pred, average='macro'),
        'recall_macro': recall_score(y_test, baseline_pred, average='macro')
    }
])

print(comparison_df)
comparison_df.to_csv('./artifacts/model_comparison.csv', index=False)
print('Saved comparison to ./artifacts/model_comparison.csv')

# Prepare one-vs-rest labels
y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))

In [ ]:
# ROC and PR curves (one-vs-rest) for CNN-LSTM and baseline

# Macro ROC-AUC
cnn_auc_per_class, base_auc_per_class = [], []
cnn_ap_per_class, base_ap_per_class = [], []

for i in range(len(classes)):
    fpr_cnn, tpr_cnn, _ = roc_curve(y_test_bin[:, i], cnn_proba[:, i])
    fpr_base, tpr_base, _ = roc_curve(y_test_bin[:, i], baseline_proba[:, i])

    cnn_auc_per_class.append(auc(fpr_cnn, tpr_cnn))
    base_auc_per_class.append(auc(fpr_base, tpr_base))

    cnn_ap_per_class.append(average_precision_score(y_test_bin[:, i], cnn_proba[:, i]))
    base_ap_per_class.append(average_precision_score(y_test_bin[:, i], baseline_proba[:, i]))

print(f'CNN-LSTM macro ROC-AUC: {np.mean(cnn_auc_per_class):.4f}')
print(f'Baseline macro ROC-AUC: {np.mean(base_auc_per_class):.4f}')
print(f'CNN-LSTM macro AP:      {np.mean(cnn_ap_per_class):.4f}')
print(f'Baseline macro AP:      {np.mean(base_ap_per_class):.4f}')

# Plot ROC for top-4 frequent classes
top_classes = np.argsort(np.bincount(y_test))[-4:]
plt.figure(figsize=(12, 5))
for i in top_classes:
    fpr_cnn, tpr_cnn, _ = roc_curve(y_test_bin[:, i], cnn_proba[:, i])
    fpr_base, tpr_base, _ = roc_curve(y_test_bin[:, i], baseline_proba[:, i])
    plt.plot(fpr_cnn, tpr_cnn, label=f'CNN {idx_to_name_ru[i]} (AUC={auc(fpr_cnn, tpr_cnn):.3f})')
    plt.plot(fpr_base, tpr_base, linestyle='--', label=f'BASE {idx_to_name_ru[i]} (AUC={auc(fpr_base, tpr_base):.3f})')

plt.plot([0, 1], [0, 1], 'k:', alpha=0.7)
plt.title('ROC one-vs-rest (топ-4 классов)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Plot PR for top-4 frequent classes
plt.figure(figsize=(12, 5))
for i in top_classes:
    p_cnn, r_cnn, _ = precision_recall_curve(y_test_bin[:, i], cnn_proba[:, i])
    p_base, r_base, _ = precision_recall_curve(y_test_bin[:, i], baseline_proba[:, i])
    ap_cnn = average_precision_score(y_test_bin[:, i], cnn_proba[:, i])
    ap_base = average_precision_score(y_test_bin[:, i], baseline_proba[:, i])
    plt.plot(r_cnn, p_cnn, label=f'CNN {idx_to_name_ru[i]} (AP={ap_cnn:.3f})')
    plt.plot(r_base, p_base, linestyle='--', label=f'BASE {idx_to_name_ru[i]} (AP={ap_base:.3f})')

plt.title('Precision-Recall one-vs-rest (топ-4 классов)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Calibration plot for selected classes
selected_classes = np.argsort(np.bincount(y_test))[-4:]

plt.figure(figsize=(12, 5))
for i in selected_classes:
    frac_pos_cnn, mean_pred_cnn = calibration_curve(y_test_bin[:, i], cnn_proba[:, i], n_bins=10, strategy='quantile')
    frac_pos_base, mean_pred_base = calibration_curve(y_test_bin[:, i], baseline_proba[:, i], n_bins=10, strategy='quantile')

    plt.plot(mean_pred_cnn, frac_pos_cnn, marker='o', label=f'CNN {idx_to_name_ru[i]}')
    plt.plot(mean_pred_base, frac_pos_base, marker='x', linestyle='--', label=f'BASE {idx_to_name_ru[i]}')

plt.plot([0, 1], [0, 1], 'k:', label='Идеальная калибровка')
plt.title('Calibration plot (one-vs-rest, топ-4 классов)')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Error case export: hardest and most uncertain windows
error_rows = []
for i in range(len(y_test)):
    true_idx = int(y_test[i])
    pred_idx = int(cnn_pred[i]) if 'cnn_pred' in globals() else int(y_test_pred[i])
    conf = float(cnn_proba[i, pred_idx]) if 'cnn_proba' in globals() else np.nan
    true_prob = float(cnn_proba[i, true_idx]) if 'cnn_proba' in globals() else np.nan

    error_rows.append({
        'sample_idx': i,
        'true_idx': true_idx,
        'true_label_ru': idx_to_name_ru[true_idx],
        'pred_idx': pred_idx,
        'pred_label_ru': idx_to_name_ru[pred_idx],
        'is_error': int(pred_idx != true_idx),
        'pred_confidence': conf,
        'true_class_probability': true_prob,
        'confidence_gap': conf - true_prob if not np.isnan(conf) and not np.isnan(true_prob) else np.nan
    })

error_df = pd.DataFrame(error_rows)
error_df_sorted = error_df.sort_values(['is_error', 'pred_confidence'], ascending=[False, True])
error_df_sorted.to_csv('./artifacts/error_cases.csv', index=False)

print('Saved error cases to ./artifacts/error_cases.csv')
print(error_df_sorted.head(10))

In [ ]:
# GroupKFold evaluation (baseline, fast and leakage-safe)
# This provides a robust cross-subject stability estimate.

def build_window_stats(X_scaled_windows: np.ndarray) -> np.ndarray:
    return np.concatenate([
        X_scaled_windows.mean(axis=1),
        X_scaled_windows.std(axis=1),
        X_scaled_windows.max(axis=1),
        X_scaled_windows.min(axis=1)
    ], axis=1)

X_stats_all = build_window_stats(np.concatenate([X_train_scaled, X_val_scaled, X_test_scaled], axis=0))
y_all = np.concatenate([y_train, y_val, y_test], axis=0)

# Reconstruct groups for all split windows
groups_all = np.concatenate([
    g_trainval[train_idx_local],
    g_trainval[val_idx_local],
    groups[test_idx]
], axis=0)

gkf = GroupKFold(n_splits=5)
cv_rows = []
for fold, (tr, te) in enumerate(gkf.split(X_stats_all, y_all, groups=groups_all), start=1):
    clf = OneVsRestClassifier(
        LogisticRegression(max_iter=1500, class_weight='balanced', random_state=SEED)
    )
    clf.fit(X_stats_all[tr], y_all[tr])
    pred = clf.predict(X_stats_all[te])

    cv_rows.append({
        'fold': fold,
        'accuracy': accuracy_score(y_all[te], pred),
        'f1_weighted': f1_score(y_all[te], pred, average='weighted'),
        'precision_macro': precision_score(y_all[te], pred, average='macro'),
        'recall_macro': recall_score(y_all[te], pred, average='macro')
    })

cv_df = pd.DataFrame(cv_rows)
cv_summary = cv_df[['accuracy', 'f1_weighted', 'precision_macro', 'recall_macro']].agg(['mean', 'std'])

cv_df.to_csv('./artifacts/groupkfold_baseline_folds.csv', index=False)
cv_summary.to_csv('./artifacts/groupkfold_baseline_summary.csv', index=True)

print('GroupKFold fold metrics:')
print(cv_df)
print('\nGroupKFold summary (mean/std):')
print(cv_summary)

In [ ]:
# Final comparison table for presentation
baseline_cv_mean = cv_summary.loc['mean']

final_compare = pd.DataFrame([
    {
        'model': 'CNN-LSTM (single subject-holdout split)',
        'accuracy': float(acc),
        'f1_weighted': float(f1_w),
        'precision_macro': float(macro_precision),
        'recall_macro': float(macro_recall)
    },
    {
        'model': 'Baseline OVR LogisticRegression (single split)',
        'accuracy': float(comparison_df.loc[comparison_df['model'] == 'Baseline OVR LogisticRegression', 'accuracy'].values[0]),
        'f1_weighted': float(comparison_df.loc[comparison_df['model'] == 'Baseline OVR LogisticRegression', 'f1_weighted'].values[0]),
        'precision_macro': float(comparison_df.loc[comparison_df['model'] == 'Baseline OVR LogisticRegression', 'precision_macro'].values[0]),
        'recall_macro': float(comparison_df.loc[comparison_df['model'] == 'Baseline OVR LogisticRegression', 'recall_macro'].values[0])
    },
    {
        'model': 'Baseline OVR LogisticRegression (GroupKFold mean)',
        'accuracy': float(baseline_cv_mean['accuracy']),
        'f1_weighted': float(baseline_cv_mean['f1_weighted']),
        'precision_macro': float(baseline_cv_mean['precision_macro']),
        'recall_macro': float(baseline_cv_mean['recall_macro'])
    }
])

final_compare.to_csv('./artifacts/final_model_comparison_for_defense.csv', index=False)
print(final_compare)
print('Saved final comparison to ./artifacts/final_model_comparison_for_defense.csv')

## Script Speech (2–3 minutes, RU)

Здравствуйте, уважаемая комиссия.

Тема моей дипломной работы — интеллектуальная система мониторинга состояния человека на основе носимых сенсоров. В качестве базы данных я использовал набор MHEALTH, который содержит многоканальные сигналы и разметку двигательной активности.

В моей реализации решается задача классификации **12 видов активности**: от статических состояний, таких как стоя, сидя и лежа, до динамических, включая ходьбу, подъем по лестнице, бег и прыжки. Для практической интерпретации я добавил русские и английские подписи классов.

С технической точки зрения пайплайн построен в несколько этапов. Сначала выполняется надежная загрузка и проверка данных, затем формируются временные окна фиксированной длины с перекрытием. Ключевой момент — разделение выборки по субъектам, что предотвращает утечку данных между обучением и тестированием и делает оценку более честной.

Далее применяется нормализация признаков, и модель обучается на **PyTorch**. Архитектура — гибрид **CNN-LSTM**: сверточные слои выделяют локальные паттерны в сенсорных сигналах, а LSTM учитывает временную динамику активности. Для устойчивого обучения используются class weights, scheduler и early stopping.

Качество оценивается по метрикам Accuracy и Weighted F1, а также по матрице ошибок и подробному отчету по каждому классу. Кроме классификации, в проекте реализован модуль обнаружения аномалий на основе Isolation Forest, который позволяет отмечать нетипичные паттерны поведения.

В результате я получил воспроизводимый и расширяемый прототип, близкий к production-подходу: модель, scaler и словари меток сохраняются как артефакты и могут быть использованы в дальнейшем для интеграции в реальную систему мониторинга.

Спасибо за внимание. Готов ответить на вопросы.